# Market Regime Detection via Data Smashing 2.0

This notebook uses **lsmash** to find **cross-temporal relationships** in financial market data.

The core idea: encode each year's daily return sequence as a stream of discrete symbols
(bear / neutral / bull days), then compute pairwise Sequence Likelihood divergences.
Years with similar *statistical structure* — not just similar returns — will cluster
together in the distance matrix.

**What to expect:**
- Crisis years (2001–02, 2008–09, 2020, 2022) should form their own cluster
- Quiet bull-market years should cluster separately
- The dendrogram reveals market *regime similarity* that simple correlation misses

**Part 1** — Annual regime comparison: S&P 500, 2000–2023  
**Part 2** — Cross-asset structural comparison: which asset classes share a common regime signature?

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import yfinance as yf
import lsmash as ls
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# Years shaded as known crisis / stress periods
CRISIS_YEARS = {2001, 2002, 2008, 2009, 2020, 2022}

def make_lsmash_opts():
    opt = ls.LsmashOptions()
    opt.data_type  = 'symbolic'
    opt.data_dir   = 'row'
    opt.sae        = True
    opt.num_repeat = 30
    opt.depth      = 8
    return opt

---
## Part 1 — Annual Regime Comparison: S&P 500 (2000–2023)

### 1a. Download and discretise

In [ ]:
print('Downloading S&P 500 daily data...')
sp500 = yf.download('^GSPC', start='2000-01-01', end='2024-01-01', progress=False)
closes = sp500['Close'].squeeze().dropna()

# Daily log returns
log_ret = np.log(closes / closes.shift(1)).dropna()
print(f'  {len(log_ret):,} trading days from {log_ret.index[0].date()} to {log_ret.index[-1].date()}')

# Discretise into 3 symbols using global tertile thresholds:
#   0 = bear day (bottom third of all returns)
#   1 = neutral  (middle third)
#   2 = bull day (top third)
q33, q67 = np.percentile(log_ret, [33.3, 66.7])
print(f'  Thresholds: bear < {q33:.4f} < neutral < {q67:.4f} < bull')

def discretise(series):
    """Returns list of unsigned ints {0, 1, 2}."""
    return np.digitize(series.values, bins=[q33, q67]).tolist()

# Split by calendar year — require at least 200 trading days
annual = {}
for year in range(2000, 2024):
    yr = log_ret[log_ret.index.year == year]
    if len(yr) >= 200:
        annual[year] = discretise(yr)

labels_yr = [str(y) for y in annual.keys()]
seqs_yr   = list(annual.values())
print(f'\n{len(seqs_yr)} annual sequences ({labels_yr[0]}–{labels_yr[-1]})')
print(f'Sequence lengths: {min(len(s) for s in seqs_yr)}–{max(len(s) for s in seqs_yr)} trading days')

### 1b. Compute pairwise SL divergence

In [ ]:
print('Running lsmash on annual S&P 500 sequences...')
D_yr = ls.from_sequences(seqs_yr, make_lsmash_opts())
print(f'Distance matrix shape: {D_yr.shape}')

### 1c. Visualise — heatmap and dendrogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# ── Heatmap ──────────────────────────────────────────────────────────────
ax = axes[0]
sns.heatmap(
    D_yr, xticklabels=labels_yr, yticklabels=labels_yr,
    cmap='magma_r', ax=ax, linewidths=0.3, linecolor='#333',
    cbar_kws={'label': 'SL Divergence'},
)
ax.set_title('S&P 500 — Annual Regime Distance Matrix', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)
# Highlight crisis years in red
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    if int(tick.get_text()) in CRISIS_YEARS:
        tick.set_color('crimson')
        tick.set_fontweight('bold')

# ── Dendrogram ────────────────────────────────────────────────────────────
ax2 = axes[1]
condensed = squareform(D_yr, checks=False)
Z = linkage(condensed, method='average')
dend = dendrogram(Z, labels=labels_yr, ax=ax2, leaf_font_size=10,
                  leaf_rotation=45, color_threshold=0, above_threshold_color='steelblue')
ax2.set_title('Hierarchical Clustering of Market Regimes (UPGMA)', fontsize=12, fontweight='bold')
ax2.set_ylabel('SL Divergence')
# Colour crisis year labels red
for tick in ax2.get_xticklabels():
    if int(tick.get_text()) in CRISIS_YEARS:
        tick.set_color('crimson')
        tick.set_fontweight('bold')

legend = [
    mpatches.Patch(color='crimson',   label='Crisis / stress year'),
    mpatches.Patch(color='steelblue', label='Other year'),
]
axes[1].legend(handles=legend, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('market_regimes_annual.png', dpi=150, bbox_inches='tight')
plt.show()

### 1d. Most similar years (nearest neighbours)

In [ ]:
df_yr = pd.DataFrame(D_yr, index=labels_yr, columns=labels_yr)

print('Top-3 most similar years to each crisis year:\n')
for cy in [y for y in labels_yr if int(y) in CRISIS_YEARS]:
    neighbors = df_yr[cy].drop(cy).nsmallest(3)
    neighbor_str = ', '.join(f"{yr} ({d:.4f})" for yr, d in neighbors.items())
    print(f'  {cy}: {neighbor_str}')

---
## Part 2 — Cross-Asset Structural Comparison

Which asset classes share the most similar regime signature over the same period?
We compare five major instruments — equities, gold, crude oil, the VIX fear gauge,
and long-term Treasury yields — each encoded as a full-history symbol stream.

### 2a. Download and discretise all assets

In [ ]:
ASSETS = {
    'S&P 500':   '^GSPC',
    'Gold':      'GC=F',
    'Crude Oil': 'CL=F',
    'VIX':       '^VIX',
    'US 10Y':    '^TNX',
}

asset_seqs, asset_labels = [], []
print('Downloading asset data (2005–2023)...')

for name, ticker in ASSETS.items():
    raw = yf.download(ticker, start='2005-01-01', end='2024-01-01',
                      progress=False, auto_adjust=True)
    prices = raw['Close'].squeeze().dropna()
    if len(prices) < 500:
        print(f'  {name}: insufficient data, skipping')
        continue
    ret = np.log(prices / prices.shift(1)).dropna()
    # Per-asset tertile thresholds (each asset has its own volatility scale)
    q33a, q67a = np.percentile(ret, [33.3, 66.7])
    sym = np.digitize(ret.values, bins=[q33a, q67a]).tolist()
    asset_seqs.append(sym)
    asset_labels.append(name)
    print(f'  {name} ({ticker}): {len(sym):,} days')

print(f'\n{len(asset_seqs)} assets ready.')

### 2b. Compute pairwise distances

In [ ]:
print('Running lsmash on cross-asset sequences...')
D_assets = ls.from_sequences(asset_seqs, make_lsmash_opts())
print(f'Distance matrix shape: {D_assets.shape}')

df_assets = pd.DataFrame(D_assets, index=asset_labels, columns=asset_labels)
print('\nPairwise SL Divergence Matrix:')
print(df_assets.round(4).to_string())

### 2c. Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(
    D_assets, annot=True, fmt='.4f', cmap='viridis',
    xticklabels=asset_labels, yticklabels=asset_labels,
    ax=axes[0], linewidths=0.5,
    cbar_kws={'label': 'SL Divergence'},
)
axes[0].set_title('Cross-Asset Regime Distance (2005–2023)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Dendrogram
condensed_a = squareform(D_assets, checks=False)
Za = linkage(condensed_a, method='average')
dendrogram(Za, labels=asset_labels, ax=axes[1], leaf_font_size=11,
           color_threshold=0, above_threshold_color='darkorange')
axes[1].set_title('Cross-Asset Hierarchical Clustering', fontsize=12, fontweight='bold')
axes[1].set_ylabel('SL Divergence')

plt.tight_layout()
plt.savefig('market_regimes_assets.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 3 — Lead-Lag Analysis: Does a Commodity Predict a Security?

Traditional correlation finds assets that move *together*. Here we ask a sharper
question: **does a commodity move first, with a specific security following days later?**

**Method:** for every lag *k* (trading days), we align the commodity sequence
against the stock sequence shifted by *k* days and compute the SL divergence:

| Lag | Comparison | Interpretation |
|-----|-----------|----------------|
| `lag = 0` | commodity[0:N] vs stock[0:N] | simultaneous |
| `lag = +k` | commodity[0:N−k] vs stock[k:N] | **commodity leads** |
| `lag = −k` | commodity[k:N] vs stock[0:N−k] | stock leads |

A minimum SL distance at **lag > 0** means the commodity's statistical structure
predicts the stock's structure *k* days later — a direction-agnostic, nonlinear
lead-lag signal.

We study **Crude Oil (CL=F)** against five energy-sector stocks, then cross-check
with **Gold (GC=F)** against gold-mining equities.

### 3a. Configuration

In [ ]:
STUDY_START = "2010-01-01"
STUDY_END   = "2024-01-01"
LAGS        = list(range(-5, 21))   # -5 (stock leads) … +20 (commodity leads)

COMMODITIES = {
    "Crude Oil": "CL=F",
    "Gold":      "GC=F",
}

EQUITIES = {
    # Energy sector — expected to follow crude oil
    "XOM":  "ExxonMobil",
    "CVX":  "Chevron",
    "OXY":  "Occidental",
    "HAL":  "Halliburton",
    "SLB":  "SLB",
    # Gold miners — expected to follow gold
    "NEM":  "Newmont",
    "GOLD": "Barrick Gold",
    "KGC":  "Kinross Gold",
}

PAIRS = [
    ("Crude Oil", "CL=F",  ["XOM", "CVX", "OXY", "HAL", "SLB"]),
    ("Gold",      "GC=F",  ["NEM", "GOLD", "KGC"]),
]

### 3b. Download and discretise all instruments

In [ ]:
all_tickers = list(COMMODITIES.values()) + list(EQUITIES.keys())
print(f"Downloading {len(all_tickers)} instruments ({STUDY_START} to {STUDY_END})...")
raw    = yf.download(all_tickers, start=STUDY_START, end=STUDY_END,
                     progress=False, auto_adjust=True)
prices = raw["Close"].dropna(how="all")
rets   = np.log(prices / prices.shift(1)).dropna()
print(f"  {len(rets):,} aligned trading days")
print(f"  Tickers: {list(rets.columns)}")

def discretise_series(series):
    q33, q67 = np.percentile(series.dropna(), [33.3, 66.7])
    return np.digitize(series.values, bins=[q33, q67]).tolist()

sym_p3 = {col: discretise_series(rets[col]) for col in rets.columns}
print("Sequence lengths:")
for k, v in sym_p3.items():
    print(f"  {k:6s}: {len(v):,} symbols")

### 3c. Lead-lag SL distance function

In [ ]:
opt_pair = ls.LsmashOptions()
opt_pair.data_type  = "symbolic"
opt_pair.data_dir   = "row"
opt_pair.sae        = True
opt_pair.num_repeat = 30
opt_pair.depth      = 8

def sld_at_lag(comm_sym, stock_sym, lag):
    """
    lag > 0: commodity leads — compare comm[0:N-lag] vs stock[lag:N]
    lag = 0: simultaneous   — compare comm[0:N]     vs stock[0:N]
    lag < 0: stock leads    — compare comm[-lag:N]  vs stock[0:N+lag]
    """
    N      = min(len(comm_sym), len(stock_sym))
    offset = abs(lag)
    if lag >= 0:
        a = comm_sym[: N - offset] if offset > 0 else comm_sym[:N]
        b = stock_sym[offset:]     if offset > 0 else stock_sym[:N]
    else:
        a = comm_sym[offset:]
        b = stock_sym[: N - offset]
    D = ls.from_sequences([list(a), list(b)], opt_pair)
    return float(D[0, 1])

### 3d. Compute SL divergence across every lag

In [ ]:
lag_results = {}   # (comm_name, stock_ticker) -> [distance per lag]

for comm_name, comm_ticker, stock_tickers in PAIRS:
    if comm_ticker not in sym_p3:
        print(f"{comm_ticker} not available, skipping"); continue
    comm_sym = sym_p3[comm_ticker]
    print(f"\n{comm_name} ({comm_ticker}) vs:")
    for ticker in stock_tickers:
        if ticker not in sym_p3:
            print(f"  {ticker}: not available"); continue
        dists   = [sld_at_lag(comm_sym, sym_p3[ticker], lag) for lag in LAGS]
        opt_lag = LAGS[int(np.argmin(dists))]
        min_d   = min(dists)
        lag_results[(comm_name, ticker)] = dists
        print(f"  {ticker:5s} ({EQUITIES[ticker]:15s}): "
              f"optimal lag = {opt_lag:+3d} days,  min SLD = {min_d:.5f}")

### 3e. Visualise — SL distance vs lag curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
palette   = plt.cm.tab10.colors

for ax_idx, (comm_name, comm_ticker, stock_tickers) in enumerate(PAIRS):
    ax = axes[ax_idx]
    for ci, ticker in enumerate(stock_tickers):
        key = (comm_name, ticker)
        if key not in lag_results: continue
        dists   = lag_results[key]
        opt_lag = LAGS[int(np.argmin(dists))]
        min_d   = min(dists)
        color   = palette[ci % len(palette)]
        label   = f"{ticker} ({EQUITIES[ticker]})  [opt lag={opt_lag:+d}d]"
        ax.plot(LAGS, dists, color=color, lw=2, label=label)
        ax.scatter([opt_lag], [min_d], color=color, s=80, zorder=5)

    ax.axvline(0, color="black", lw=1, ls="--", alpha=0.5)
    ax.axvspan(0, max(LAGS),   alpha=0.06, color="green",  label="commodity leads ->")
    ax.axvspan(min(LAGS), 0,   alpha=0.06, color="orange", label="<- stock leads")
    ax.set_xlabel("Lag (trading days)   <-stock leads | commodity leads->", fontsize=10)
    ax.set_ylabel("SL Divergence", fontsize=10)
    ax.set_title(f"{comm_name} -> equities: SL distance vs temporal lag",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("lead_lag_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: examples/lead_lag_analysis.png")

### 3f. Summary table

In [ ]:
rows = []
for (comm_name, ticker), dists in lag_results.items():
    opt_lag     = LAGS[int(np.argmin(dists))]
    min_d       = min(dists)
    d_at_0      = dists[LAGS.index(0)]
    improvement = (d_at_0 - min_d) / d_at_0 * 100
    direction   = (
        f"Commodity leads by {opt_lag}d" if opt_lag > 0
        else "Simultaneous" if opt_lag == 0
        else f"Stock leads by {-opt_lag}d"
    )
    rows.append({
        "Commodity":     comm_name,
        "Security":      f"{ticker} ({EQUITIES[ticker]})",
        "Optimal lag":   opt_lag,
        "Min SLD":       round(min_d, 5),
        "SLD at lag=0":  round(d_at_0, 5),
        "Improvement %": round(improvement, 1),
        "Direction":     direction,
    })

summary = pd.DataFrame(rows).sort_values(["Commodity", "Optimal lag"])
print(summary.to_string(index=False))

print("\n--- Pairs where commodity leads (lag > 0) ---")
leading = summary[summary["Optimal lag"] > 0]
if len(leading):
    print(leading[["Commodity", "Security", "Optimal lag",
                   "Min SLD", "Improvement %", "Direction"]].to_string(index=False))
else:
    print("None found — try increasing num_repeat or widening the lag range.")